In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv

In [12]:
load_dotenv()
# llm_model = ChatOpenRouter(model="meta-llama/llama-3-8b-instruct")
llm_model = ChatOpenRouter(model="openai/gpt-oss-120b:free")

In [13]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

In [14]:
def generate_outline(state: BlogState) -> BlogState:
    topic = state['topic']

    prompt = f'''
You're a professional blog writer. Give a blog outline in around 5 words for the below topic.
Topic: {topic}
'''
    
    outline = llm_model.invoke(prompt).content

    state['outline'] = outline
    
    return state

In [15]:
def generate_blog(state: BlogState) -> BlogState:
    topic = state['topic']

    prompt = f'''
You're an experienced blog writer. You need to write a blog in around 100 words for the below topic.
Topic: {topic}
'''
    
    content = llm_model.invoke(prompt).content

    state['content'] = content
    
    return state

In [16]:
graph = StateGraph(BlogState)

graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog', generate_blog)

graph.add_edge(START, 'generate_blog')
graph.add_edge('generate_outline', 'generate_blog')
graph.add_edge('generate_blog', END)

workflow = graph.compile()

In [17]:
input_state = {'topic': 'Artificial Intelligence'}
output_state = workflow.invoke(input_state)

print(output_state['content'])

Artificial Intelligence—once the realm of sci‑fi fantasies—is now the quiet engine driving everything from personalized playlists to life‑saving medical diagnostics. At its core, AI mimics human learning: algorithms ingest massive data sets, spot patterns, and make predictions faster and more accurately than any human could. This capability is reshaping industries, slashing costs, and unlocking innovations such as self‑driving cars and real‑time language translation. Yet the rapid ascent also raises ethical questions about bias, privacy, and job displacement. Embracing AI responsibly means fostering transparency, continuous oversight, and upskilling the workforce. When guided wisely, AI promises not just smarter machines, but a smarter, more inclusive future.
